In [1]:
import sys
import os

import torch
import torch.nn

import numpy as np

In [3]:
import torch.nn.functional as F

# !python -m pip install ultralytics

# print("PyTorch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("cuDNN Version:", torch.backends.cudnn.version())

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super(CNN, self).init()
        self.convLayer1 = nn.conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1)
        self.convLayer2 = nn.conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1)

        

In [ ]:
class HeatMapNet(nn.Module):
    def __init__(self):
        super(CNN, self).init()
        self.initialConv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1)
        self.

In [ ]:
class BackboneNet2_5(nn.Module):
    def __init__(self):
        super().__init__()
        self.initialConv = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels = 256, kernel_size = 3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        self.EncodeBlocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2),
                nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
                nn.ReLU()
            ) for _ in range(5)])
        
        self.DecodeBlocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1,stride=1),
                nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
                #F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            ) for _ in range(5)])
        
        self.UBlock = nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
            #F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        )
        
        self.HeatConv = nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=7, stride=1, padding=3),
            nn.ReLU(),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=7, stride=1, padding=3),
            nn.ReLU(),
            nn.Conv2d(in_channels=256, out_channels = 42, kernel_size=7, stride=1, padding=3),
        )
        
        #self.EncodeBlocks = nn.ModuleList([self.EncodeBlock.deepcopy() for _ in range(5)])
        
        #self.DecodeBlocks = nn.ModuleList([self.DecodeBlock.deepcopy() for _ in range(5)])
    def forward(self, x):
        skipBlocks = []
        
        x = self.initialConv(x)
        
        for block in self.EncodeBlocks:
            skipBlocks.append(x.clone())
            x = block(x)
            
        
        x = self.UBlock(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        
        for i in range(5):
            inputX = torch.cat([x, skipBlocks[-1 * i - 1]], dim=1)
            x = self.DecodeBlocks[i](inputX)
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            
        return self.HeatConv(x)

In [ ]:
class Latent2_5(nn.Module):
    def __init__(self):
        self.backbone = BackboneNet2_5()
        
    def forward(self, x):
        Heatmaps = self.backbone(x)
        
        HeatZ = Heatmaps[, 21:]
        HeatXY = Heatmaps[, :21]
        
        
    